# Описательные статистики: датасет NMES1988

**Дисциплина:** Введение в анализ больших данных  
**Задание 3**

Разбираем описательные статистики на реальном датасете **NMES1988** (медицинское обследование, 4406 наблюдений, 19 переменных).

**Инструменты:** `pandas`, `numpy`, `scipy.stats`, `matplotlib`, `seaborn`.

## 0. Импорт библиотек и настройка

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

---
## 1–2. Загрузка и знакомство с данными

Файл `NMES1988.csv` лежит рядом с тетрадью. Первая колонка — `rownames`, её убираем через `index_col=0`.

In [ ]:
data = pd.read_csv("NMES1988.csv", index_col=0)

# Оставляем только переменные, нужные по заданию
cols = ["visits", "health", "chronic", "adl", "region", "age",
        "gender", "married", "school", "income", "employed", "insurance"]
data = data[cols]

print("Размерность:", data.shape)
print("Столбцы:", data.columns.tolist())
data.head()

In [ ]:
data.describe(include="all")

---
## 3. Гистограмма visits + кривая плотности

In [ ]:
visits = data["visits"]

fig, ax = plt.subplots()
ax.hist(visits, bins=40, density=True, color="steelblue",
        edgecolor="white", alpha=0.7, label="гистограмма")

kde = stats.gaussian_kde(visits.dropna())
xs = np.linspace(visits.min(), visits.max(), 300)
ax.plot(xs, kde(xs), color="crimson", lw=2, label="KDE")

ax.set_title("Распределение visits (NMES1988)")
ax.set_xlabel("visits")
ax.set_ylabel("плотность")
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Среднее и стандартная ошибка среднего (SE)

In [ ]:
n = visits.count()
mean_v = visits.mean()
sd_v = visits.std(ddof=1)
se_v = sd_v / np.sqrt(n)

print(f"n     = {n}")
print(f"mean  = {mean_v:.4f}")
print(f"sd    = {sd_v:.4f}")
print(f"SE    = {se_v:.4f}")

---
## 5. Медиана, квартили, summary

In [ ]:
print("Медиана:", visits.median())
print("Квантили 0%, 25%, 50%, 75%, 100%:")
print(visits.quantile([0, 0.25, 0.5, 0.75, 1.0]))
print()
print(visits.describe())

---
## 6. Асимметрия и эксцесс

In [ ]:
sk = stats.skew(visits, bias=False)
ku = stats.kurtosis(visits, bias=False)
print(f"skewness = {sk:.4f}")
print(f"kurtosis (excess) = {ku:.4f}")

---
## 7. Медианы и квартили числовых переменных по полу

In [ ]:
numeric_cols = ["visits", "chronic", "age", "school", "income"]

for col in numeric_cols:
    print(f"--- {col} ---")
    print(data.groupby("gender")[col].describe().round(3))
    print()

---
## 8. Диаграмма размаха visits по полу

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=data, x="gender", y="visits", hue="gender",
            palette="pastel", legend=False)
plt.title("Диаграмма размаха visits по полу")
plt.xlabel("gender")
plt.ylabel("visits")
plt.tight_layout()
plt.show()

---
## 9. Медианы visits по gender × region

In [ ]:
med_gr = data.groupby(["gender", "region"])["visits"].median().unstack()
med_gr.round(3)

---
## 10. Столбиковые диаграммы медиан visits по gender и region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

med_gender = data.groupby("gender")["visits"].median()
med_gender.plot(kind="bar", ax=axes[0], color="teal", edgecolor="black")
axes[0].set_title("Медиана visits по gender")
axes[0].set_ylabel("Медиана visits")
axes[0].tick_params(axis="x", rotation=0)

med_region = data.groupby("region")["visits"].median().sort_values(ascending=False)
med_region.plot(kind="bar", ax=axes[1], color="coral", edgecolor="black")
axes[1].set_title("Медиана visits по region")
axes[1].set_ylabel("Медиана visits")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
med_gr.plot(kind="bar", edgecolor="black")
plt.title("Медиана visits: gender × region")
plt.ylabel("Медиана visits")
plt.xlabel("gender")
plt.legend(title="region")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 11. Медиана возраста по married — общая и по полу

In [ ]:
med_age_all = data.groupby("married")["age"].median()
print("По всей выборке:")
print(med_age_all.round(3))

med_age_sex = data.groupby(["gender", "married"])["age"].median().unstack()
print("\nПо полу:")
print(med_age_sex.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

med_age_all.plot(kind="bar", ax=axes[0], color="mediumseagreen",
                 edgecolor="black")
axes[0].set_title("Медиана age по married (вся выборка)")
axes[0].set_ylabel("Медиана age")
axes[0].tick_params(axis="x", rotation=0)

med_age_sex.plot(kind="bar", ax=axes[1], edgecolor="black")
axes[1].set_title("Медиана age: gender × married")
axes[1].set_ylabel("Медиана age")
axes[1].legend(title="gender")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

### Интерпретация пункта 11

- Женатые респонденты в среднем старше неженатых (медиана выше).
- Разница сохраняется и внутри каждого пола.
- Медиана возраста у женщин в категории `married=yes` и `married=no` может отличаться от мужской — смотрим на barplot.